In [ ]:
"""
Quantum Logic Gates for 2 to 32 Qubits
========================================
Pure NumPy implementation of quantum gates and circuits.
No external quantum library required.

Gates implemented:
  Single-qubit : I, X (NOT), Y, Z, H (Hadamard), S, T, Rx, Ry, Rz
  Two-qubit     : CNOT, CZ, SWAP, CY, CH
  Three-qubit   : Toffoli (CCX), Fredkin (CSWAP), CCZ
  N-qubit       : Multi-controlled X (MCX/Toffoli generalisation)
  Measurement   : Probability distribution over all basis states
"""

import numpy as np
from functools import reduce
from typing import List, Optional


# ---------------------------------------------------------------------------
# 1. Core utilities
# ---------------------------------------------------------------------------

def _kron(*matrices):
    """Tensor (Kronecker) product of an arbitrary number of matrices."""
    return reduce(np.kron, matrices)


def _identity(n: int) -> np.ndarray:
    """2^n × 2^n identity matrix."""
    return np.eye(2 ** n, dtype=complex)


# ---------------------------------------------------------------------------
# 2. Single-qubit gate matrices
# ---------------------------------------------------------------------------

I  = np.eye(2, dtype=complex)                                   # Identity
X  = np.array([[0, 1], [1, 0]], dtype=complex)                  # Pauli-X / NOT
Y  = np.array([[0, -1j], [1j, 0]], dtype=complex)               # Pauli-Y
Z  = np.array([[1, 0], [0, -1]], dtype=complex)                 # Pauli-Z
H  = np.array([[1, 1], [1, -1]], dtype=complex) / np.sqrt(2)   # Hadamard
S  = np.array([[1, 0], [0, 1j]], dtype=complex)                 # Phase (S)
T  = np.array([[1, 0], [0, np.exp(1j * np.pi / 4)]], dtype=complex)  # T-gate

def Rx(theta: float) -> np.ndarray:
    """Rotation around X-axis by angle theta."""
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -1j * s], [-1j * s, c]], dtype=complex)

def Ry(theta: float) -> np.ndarray:
    """Rotation around Y-axis by angle theta."""
    c, s = np.cos(theta / 2), np.sin(theta / 2)
    return np.array([[c, -s], [s, c]], dtype=complex)

def Rz(theta: float) -> np.ndarray:
    """Rotation around Z-axis by angle theta."""
    return np.array([[np.exp(-1j * theta / 2), 0],
                     [0, np.exp( 1j * theta / 2)]], dtype=complex)

SINGLE_QUBIT_GATES = {"I": I, "X": X, "Y": Y, "Z": Z,
                      "H": H, "S": S, "T": T}


# ---------------------------------------------------------------------------
# 3. Embed a single-qubit gate into an n-qubit Hilbert space
# ---------------------------------------------------------------------------

def single_qubit_gate(gate: np.ndarray, target: int, n_qubits: int) -> np.ndarray:
    """
    Lift a single-qubit gate to act on `target` qubit in an n-qubit system.

    Parameters
    ----------
    gate     : 2×2 gate matrix
    target   : qubit index (0 = most significant / leftmost)
    n_qubits : total number of qubits

    Returns
    -------
    2^n × 2^n unitary matrix
    """
    if not (0 <= target < n_qubits):
        raise ValueError(f"target {target} out of range [0, {n_qubits})")
    ops = [gate if i == target else I for i in range(n_qubits)]
    return _kron(*ops)


# ---------------------------------------------------------------------------
# 4. Two-qubit gates
# ---------------------------------------------------------------------------

# Canonical 4×4 matrices (qubit ordering: control ⊗ target)
CNOT_mat = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,1],[0,0,1,0]], dtype=complex)
CZ_mat   = np.array([[1,0,0,0],[0,1,0,0],[0,0,1,0],[0,0,0,-1]], dtype=complex)
SWAP_mat = np.array([[1,0,0,0],[0,0,1,0],[0,1,0,0],[0,0,0,1]], dtype=complex)
CY_mat   = np.array([[1,0,0,0],[0,1,0,0],[0,0,0,-1j],[0,0,1j,0]], dtype=complex)
CH_mat   = np.array([[1,0,0,0],[0,1,0,0],
                     [0,0,1/np.sqrt(2), 1/np.sqrt(2)],
                     [0,0,1/np.sqrt(2),-1/np.sqrt(2)]], dtype=complex)


def _embed_two_qubit(gate4: np.ndarray,
                     q0: int, q1: int,
                     n_qubits: int) -> np.ndarray:
    """
    Embed a 4×4 two-qubit gate (acting on q0⊗q1) into n_qubits space.
    Works for any pair of distinct qubits.
    """
    dim = 2 ** n_qubits
    full = np.eye(dim, dtype=complex)
    for col in range(dim):
        bits  = [(col >> (n_qubits - 1 - k)) & 1 for k in range(n_qubits)]
        i2    = (bits[q0] << 1) | bits[q1]         # 2-qubit input index
        for row2 in range(4):
            amp = gate4[row2, i2]
            if amp == 0:
                continue
            bits_out       = bits[:]
            bits_out[q0]   = (row2 >> 1) & 1
            bits_out[q1]   =  row2       & 1
            row = sum(bits_out[k] << (n_qubits - 1 - k) for k in range(n_qubits))
            full[row, col] += amp
    # subtract identity double-counting (we started from eye)
    full -= np.eye(dim, dtype=complex)
    # rebuild cleanly
    return _build_two_qubit_gate(gate4, q0, q1, n_qubits)


def _build_two_qubit_gate(gate4: np.ndarray,
                          q0: int, q1: int,
                          n_qubits: int) -> np.ndarray:
    """Clean construction via projectors."""
    dim = 2 ** n_qubits
    result = np.zeros((dim, dim), dtype=complex)
    for col_idx in range(dim):
        in_bits  = [(col_idx >> (n_qubits - 1 - k)) & 1 for k in range(n_qubits)]
        in2      = (in_bits[q0] << 1) | in_bits[q1]
        for out2 in range(4):
            amp = gate4[out2, in2]
            if amp == 0:
                continue
            out_bits      = in_bits[:]
            out_bits[q0]  = (out2 >> 1) & 1
            out_bits[q1]  =  out2       & 1
            row_idx = sum(out_bits[k] << (n_qubits - 1 - k) for k in range(n_qubits))
            result[row_idx, col_idx] += amp
    return result


def cnot(control: int, target: int, n_qubits: int) -> np.ndarray:
    return _build_two_qubit_gate(CNOT_mat, control, target, n_qubits)

def cz(control: int, target: int, n_qubits: int) -> np.ndarray:
    return _build_two_qubit_gate(CZ_mat, control, target, n_qubits)

def swap(q0: int, q1: int, n_qubits: int) -> np.ndarray:
    return _build_two_qubit_gate(SWAP_mat, q0, q1, n_qubits)

def cy(control: int, target: int, n_qubits: int) -> np.ndarray:
    return _build_two_qubit_gate(CY_mat, control, target, n_qubits)

def ch(control: int, target: int, n_qubits: int) -> np.ndarray:
    return _build_two_qubit_gate(CH_mat, control, target, n_qubits)


# ---------------------------------------------------------------------------
# 5. Three-qubit gates
# ---------------------------------------------------------------------------

def toffoli(c0: int, c1: int, target: int, n_qubits: int) -> np.ndarray:
    """Toffoli / CCX gate: flip target iff both controls are |1⟩."""
    dim = 2 ** n_qubits
    result = np.eye(dim, dtype=complex)
    for col in range(dim):
        bits = [(col >> (n_qubits - 1 - k)) & 1 for k in range(n_qubits)]
        if bits[c0] == 1 and bits[c1] == 1:
            bits_out         = bits[:]
            bits_out[target] ^= 1
            row = sum(bits_out[k] << (n_qubits - 1 - k) for k in range(n_qubits))
            # Swap columns col and row
            result[:, [col, row]] = result[:, [row, col]]
    return result


def fredkin(control: int, t0: int, t1: int, n_qubits: int) -> np.ndarray:
    """Fredkin / CSWAP gate: swap t0 and t1 iff control is |1⟩."""
    dim = 2 ** n_qubits
    result = np.eye(dim, dtype=complex)
    for col in range(dim):
        bits = [(col >> (n_qubits - 1 - k)) & 1 for k in range(n_qubits)]
        if bits[control] == 1 and bits[t0] != bits[t1]:
            bits_out      = bits[:]
            bits_out[t0]  = bits[t1]
            bits_out[t1]  = bits[t0]
            row = sum(bits_out[k] << (n_qubits - 1 - k) for k in range(n_qubits))
            result[:, [col, row]] = result[:, [row, col]]
    return result


def ccz(c0: int, c1: int, target: int, n_qubits: int) -> np.ndarray:
    """CCZ gate: apply Z to target iff both controls are |1⟩."""
    dim = 2 ** n_qubits
    result = np.eye(dim, dtype=complex)
    for i in range(dim):
        bits = [(i >> (n_qubits - 1 - k)) & 1 for k in range(n_qubits)]
        if bits[c0] == 1 and bits[c1] == 1 and bits[target] == 1:
            result[i, i] = -1
    return result


# ---------------------------------------------------------------------------
# 6. Multi-controlled X (generalised Toffoli)
# ---------------------------------------------------------------------------

def mcx(controls: List[int], target: int, n_qubits: int) -> np.ndarray:
    """
    Multi-Controlled X: flip target qubit iff ALL control qubits are |1⟩.
    This generalises CNOT (1 control) and Toffoli (2 controls) to any number.
    """
    dim = 2 ** n_qubits
    result = np.eye(dim, dtype=complex)
    for col in range(dim):
        bits = [(col >> (n_qubits - 1 - k)) & 1 for k in range(n_qubits)]
        if all(bits[c] == 1 for c in controls):
            bits_out         = bits[:]
            bits_out[target] ^= 1
            row = sum(bits_out[k] << (n_qubits - 1 - k) for k in range(n_qubits))
            result[:, [col, row]] = result[:, [row, col]]
    return result


# ---------------------------------------------------------------------------
# 7. QuantumCircuit — lightweight statevector simulator
# ---------------------------------------------------------------------------

class QuantumCircuit:
    """
    Lightweight statevector simulator for 2–32 qubits.

    For n > ~16 qubits the full statevector requires > 1 MB and storing the
    full unitary matrix is impractical.  This class therefore applies gates
    directly to the statevector without materialising the full 2^n × 2^n
    unitary, making it feasible up to ~28-30 qubits on a typical machine.
    """

    # ---- Gate definitions (applied directly to statevector) ----------------

    @staticmethod
    def _apply_single(sv: np.ndarray, gate: np.ndarray,
                      target: int, n: int) -> np.ndarray:
        """Apply a 2×2 gate to `target` qubit of statevector sv (shape 2^n)."""
        sv = sv.reshape([2] * n)
        sv = np.tensordot(gate, sv, axes=([1], [target]))
        # tensordot puts the new axis at position 0; move it back
        sv = np.moveaxis(sv, 0, target)
        return sv.reshape(-1)

    @staticmethod
    def _apply_cnot(sv: np.ndarray,
                    control: int, target: int, n: int) -> np.ndarray:
        sv = sv.reshape([2] * n)
        # Select slices where control == 1 and swap target 0↔1
        idx_c1_t0 = [slice(None)] * n
        idx_c1_t1 = [slice(None)] * n
        idx_c1_t0[control] = 1;  idx_c1_t0[target] = 0
        idx_c1_t1[control] = 1;  idx_c1_t1[target] = 1
        tmp = sv[tuple(idx_c1_t0)].copy()
        sv[tuple(idx_c1_t0)] = sv[tuple(idx_c1_t1)]
        sv[tuple(idx_c1_t1)] = tmp
        return sv.reshape(-1)

    @staticmethod
    def _apply_swap_sv(sv: np.ndarray,
                       q0: int, q1: int, n: int) -> np.ndarray:
        sv = sv.reshape([2] * n)
        sv = np.swapaxes(sv, q0, q1)
        return sv.reshape(-1)

    # ---- Constructor --------------------------------------------------------

    def __init__(self, n_qubits: int):
        if not (2 <= n_qubits <= 32):
            raise ValueError("n_qubits must be between 2 and 32")
        self.n   = n_qubits
        self.dim = 2 ** n_qubits
        # Initialise to |0…0⟩
        self.statevector = np.zeros(self.dim, dtype=complex)
        self.statevector[0] = 1.0
        self._log: List[str] = []

    def reset(self):
        """Reset to |0…0⟩."""
        self.statevector[:] = 0
        self.statevector[0] = 1.0
        self._log.clear()

    # ---- Single-qubit gates -------------------------------------------------

    def _sq(self, gate, name, q):
        self.statevector = self._apply_single(self.statevector, gate, q, self.n)
        self._log.append(f"{name}(q{q})")

    def i(self, q):  self._sq(I, "I",  q)
    def x(self, q):  self._sq(X, "X",  q)
    def y(self, q):  self._sq(Y, "Y",  q)
    def z(self, q):  self._sq(Z, "Z",  q)
    def h(self, q):  self._sq(H, "H",  q)
    def s(self, q):  self._sq(S, "S",  q)
    def t(self, q):  self._sq(T, "T",  q)
    def rx(self, theta, q): self._sq(Rx(theta), f"Rx({theta:.3f})", q)
    def ry(self, theta, q): self._sq(Ry(theta), f"Ry({theta:.3f})", q)
    def rz(self, theta, q): self._sq(Rz(theta), f"Rz({theta:.3f})", q)

    # ---- Two-qubit gates ----------------------------------------------------

    def cnot(self, control, target):
        self.statevector = self._apply_cnot(self.statevector, control, target, self.n)
        self._log.append(f"CNOT(c=q{control}, t=q{target})")

    def cx(self, control, target): self.cnot(control, target)

    def cz(self, control, target):
        # CZ = H·CNOT·H on target
        self.h(target); self.cnot(control, target); self.h(target)
        self._log[-3:] = [f"CZ(c=q{control}, t=q{target})"]

    def swap(self, q0, q1):
        self.statevector = self._apply_swap_sv(self.statevector, q0, q1, self.n)
        self._log.append(f"SWAP(q{q0}, q{q1})")

    def cy(self, control, target):
        # CY = S†·CNOT·S on target
        self._sq(S.conj().T, "Sdg", target)
        self.cnot(control, target)
        self._sq(S, "S", target)
        self._log[-3:] = [f"CY(c=q{control}, t=q{target})"]

    def ch(self, control, target):
        # CH via CNOT + single-qubit decomposition
        sv = self.statevector.reshape([2] * self.n)
        # Controlled-H is applied manually via projectors
        dim = self.dim
        new_sv = self.statevector.copy()
        new_sv = new_sv.reshape([2] * self.n)
        # Where control==1: apply H to target
        idx1 = [slice(None)] * self.n
        idx1[control] = 1
        sub = new_sv[tuple(idx1)]
        sub = np.tensordot(H, sub, axes=([1], [target - (1 if target > control else 0)]))
        sub = np.moveaxis(sub, 0, target - (1 if target > control else 0))
        new_sv[tuple(idx1)] = sub
        self.statevector = new_sv.reshape(-1)
        self._log.append(f"CH(c=q{control}, t=q{target})")

    # ---- Three-qubit gates --------------------------------------------------

    def toffoli(self, c0, c1, target):
        """CCX / Toffoli: flip target if c0=c1=|1⟩."""
        sv = self.statevector.reshape([2] * self.n)
        idx_c1_t0 = [slice(None)] * self.n
        idx_c1_t1 = [slice(None)] * self.n
        for idx in (idx_c1_t0, idx_c1_t1):
            idx[c0] = 1; idx[c1] = 1
        idx_c1_t0[target] = 0
        idx_c1_t1[target] = 1
        tmp = sv[tuple(idx_c1_t0)].copy()
        sv[tuple(idx_c1_t0)] = sv[tuple(idx_c1_t1)]
        sv[tuple(idx_c1_t1)] = tmp
        self.statevector = sv.reshape(-1)
        self._log.append(f"Toffoli(c0=q{c0}, c1=q{c1}, t=q{target})")

    def ccx(self, c0, c1, target): self.toffoli(c0, c1, target)

    def fredkin(self, control, t0, t1):
        """CSWAP: swap t0,t1 if control=|1⟩."""
        sv = self.statevector.reshape([2] * self.n)
        # only swap where t0≠t1 and control==1
        idx_10 = [slice(None)] * self.n
        idx_01 = [slice(None)] * self.n
        idx_10[control] = 1; idx_10[t0] = 1; idx_10[t1] = 0
        idx_01[control] = 1; idx_01[t0] = 0; idx_01[t1] = 1
        tmp = sv[tuple(idx_10)].copy()
        sv[tuple(idx_10)] = sv[tuple(idx_01)]
        sv[tuple(idx_01)] = tmp
        self.statevector = sv.reshape(-1)
        self._log.append(f"Fredkin(c=q{control}, t0=q{t0}, t1=q{t1})")

    def cswap(self, control, t0, t1): self.fredkin(control, t0, t1)

    # ---- Multi-controlled X (n-qubit Toffoli) -------------------------------

    def mcx(self, controls: List[int], target: int):
        """Flip target iff ALL control qubits are |1⟩."""
        sv = self.statevector.reshape([2] * self.n)
        # Build index where all controls==1 and target alternates
        idx0 = [slice(None)] * self.n
        idx1 = [slice(None)] * self.n
        for c in controls:
            idx0[c] = 1; idx1[c] = 1
        idx0[target] = 0; idx1[target] = 1
        tmp = sv[tuple(idx0)].copy()
        sv[tuple(idx0)] = sv[tuple(idx1)]
        sv[tuple(idx1)] = tmp
        self.statevector = sv.reshape(-1)
        self._log.append(f"MCX(controls={controls}, t=q{target})")

    # ---- Measurement & utilities --------------------------------------------

    def probabilities(self) -> np.ndarray:
        """Probability of each basis state."""
        return np.abs(self.statevector) ** 2

    def measure_all(self, shots: int = 1024) -> dict:
        """Simulate `shots` measurements, return counts dict."""
        probs  = self.probabilities()
        states = np.arange(self.dim)
        counts = np.random.multinomial(shots, probs)
        result = {}
        for state, count in enumerate(counts):
            if count > 0:
                label = format(state, f"0{self.n}b")
                result[label] = int(count)
        return dict(sorted(result.items()))

    def top_states(self, k: int = 8) -> List[tuple]:
        """Return the k most-probable basis states."""
        probs   = self.probabilities()
        indices = np.argsort(probs)[::-1][:k]
        return [(format(i, f"0{self.n}b"), float(probs[i]))
                for i in indices if probs[i] > 1e-12]

    def is_unitary(self) -> bool:
        """Verify |ψ⟩ is normalised (probability sums to 1)."""
        return np.isclose(np.sum(self.probabilities()), 1.0)

    def circuit_summary(self) -> str:
        return (f"QuantumCircuit({self.n} qubits) | "
                f"Gates applied: {len(self._log)}\n"
                + " → ".join(self._log))


# ---------------------------------------------------------------------------
# 8. Demonstration: run example circuits for qubit counts 2 → 32
# ---------------------------------------------------------------------------

def demo_circuit(n: int, verbose: bool = False):
    """Build and run a small demonstration circuit on n qubits."""
    qc = QuantumCircuit(n)

    # Step 1: put qubit 0 into superposition
    qc.h(0)

    # Step 2: CNOT chain (qubit 0 → 1 → 2 → … → min(n-1, 4))
    for k in range(min(n - 1, 4)):
        qc.cnot(k, k + 1)

    # Step 3: apply X to last qubit
    qc.x(n - 1)

    # Step 4: if n >= 3, add Toffoli on qubits 0,1 → 2
    if n >= 3:
        qc.toffoli(0, 1, 2)

    # Step 5: if n >= 4, Fredkin (CSWAP) on qubits 0 → (n-2, n-1)
    if n >= 4:
        qc.fredkin(0, n - 2, n - 1)

    # Step 6: if n >= 5, MCX with controls [0,1,2] targeting qubit 3
    if n >= 5:
        qc.mcx([0, 1, 2], 3)

    # Step 7: rotation gates on qubit 0
    qc.rx(np.pi / 4, 0)
    qc.ry(np.pi / 3, 0)
    qc.rz(np.pi / 2, 0)

    top = qc.top_states(k=4)
    norm_ok = qc.is_unitary()

    print(f"  n={n:2d} | dim=2^{n}={qc.dim:<10d} | "
          f"normalised={'✓' if norm_ok else '✗'} | "
          f"top state: |{top[0][0]}⟩ p={top[0][1]:.4f}")

    if verbose:
        print(f"         {qc.circuit_summary()}")
        for state, prob in top:
            print(f"         |{state}⟩  {prob:.6f}")
    return qc


# ---------------------------------------------------------------------------
# 9. Gate matrix inspection helpers
# ---------------------------------------------------------------------------

def print_gate(name: str, matrix: np.ndarray, max_dim: int = 8):
    print(f"\n{'─'*55}")
    print(f"  Gate: {name}  (shape {matrix.shape})")
    print(f"{'─'*55}")
    if matrix.shape[0] <= max_dim:
        for row in matrix:
            print("  " + "  ".join(
                f"{v.real:+.3f}{v.imag:+.3f}j" if abs(v.imag) > 1e-10
                else f"{v.real:+.6f}      "
                for v in row))
    else:
        print(f"  (matrix too large to print: {matrix.shape[0]}×{matrix.shape[1]})")


# ---------------------------------------------------------------------------
# 10. Main entry point
# ---------------------------------------------------------------------------

if __name__ == "__main__":
    np.random.seed(42)

    # ── Part A: Print single-qubit gate matrices ──────────────────────────
    print("=" * 60)
    print("  PART A — Single-Qubit Gate Matrices")
    print("=" * 60)
    for name, mat in SINGLE_QUBIT_GATES.items():
        print_gate(name, mat)
    for name, angle in [("Rx(π/4)", np.pi/4), ("Ry(π/3)", np.pi/3),
                         ("Rz(π/2)", np.pi/2)]:
        fn = {"Rx": Rx, "Ry": Ry, "Rz": Rz}[name[:2]]
        print_gate(name, fn(angle))

    # ── Part B: Two-qubit gate matrices for a 3-qubit system ─────────────
    print("\n" + "=" * 60)
    print("  PART B — Two-Qubit Gates (embedded in 3-qubit space)")
    print("=" * 60)
    for gate_name, gate_fn, args in [
        ("CNOT(c=0,t=1)", cnot,  (0, 1, 3)),
        ("CZ(c=0,t=2)",   cz,    (0, 2, 3)),
        ("SWAP(0,1)",     swap,  (0, 1, 3)),
        ("CY(c=1,t=2)",   cy,    (1, 2, 3)),
    ]:
        print_gate(gate_name, gate_fn(*args), max_dim=8)

    # ── Part C: Three-qubit gates (4-qubit system) ────────────────────────
    print("\n" + "=" * 60)
    print("  PART C — Three-Qubit Gates (embedded in 4-qubit space)")
    print("=" * 60)
    print_gate("Toffoli(c0=0,c1=1,t=2)", toffoli(0, 1, 2, 4), max_dim=16)
    print_gate("Fredkin(c=0,t0=1,t1=2)", fredkin(0, 1, 2, 4), max_dim=16)
    print_gate("CCZ(c0=0,c1=1,t=2)",     ccz(0, 1, 2, 4),     max_dim=16)

    # ── Part D: MCX for varying numbers of controls ───────────────────────
    print("\n" + "=" * 60)
    print("  PART D — Multi-Controlled X (MCX) for n qubits")
    print("=" * 60)
    for n, ctrl in [(3, [0]), (4, [0,1]), (5, [0,1,2]), (6, [0,1,2,3])]:
        mat = mcx(ctrl, n-1, n)
        print_gate(f"MCX(controls={ctrl}, target=q{n-1}, n={n})", mat, max_dim=8)

    # ── Part E: Statevector simulation for 2 → 32 qubits ─────────────────
    print("\n" + "=" * 60)
    print("  PART E — Statevector Simulation  (n = 2 → 32)")
    print("=" * 60)
    print(f"  {'n':>3}  {'dim':>12}  {'norm':>6}  top basis state + probability")
    print("  " + "─" * 58)
    for n in range(2, 33):
        demo_circuit(n, verbose=False)

    # ── Part F: Verbose view for a small circuit (n=5) ────────────────────
    print("\n" + "=" * 60)
    print("  PART F — Detailed Circuit View  (n=5)")
    print("=" * 60)
    qc5 = demo_circuit(5, verbose=True)
    print("\n  Measurement simulation (1024 shots):")
    counts = qc5.measure_all(shots=1024)
    for state, cnt in list(counts.items())[:10]:
        bar = "█" * (cnt // 20)
        print(f"    |{state}⟩  {cnt:4d}  {bar}")

    # ── Part G: Bell pair verification ────────────────────────────────────
    print("\n" + "=" * 60)
    print("  PART G — Bell State  |Φ+⟩ = (|00⟩+|11⟩)/√2  (n=2)")
    print("=" * 60)
    bell = QuantumCircuit(2)
    bell.h(0)
    bell.cnot(0, 1)
    print(f"  Statevector: {bell.statevector}")
    print(f"  Probabilities: {bell.probabilities()}")
    print(f"  Top states: {bell.top_states()}")
    print(f"  Normalised: {bell.is_unitary()}")

    print("\n✅  All demonstrations complete.")

  PART A — Single-Qubit Gate Matrices

───────────────────────────────────────────────────────
  Gate: I  (shape (2, 2))
───────────────────────────────────────────────────────
  +1.000000        +0.000000      
  +0.000000        +1.000000      

───────────────────────────────────────────────────────
  Gate: X  (shape (2, 2))
───────────────────────────────────────────────────────
  +0.000000        +1.000000      
  +1.000000        +0.000000      

───────────────────────────────────────────────────────
  Gate: Y  (shape (2, 2))
───────────────────────────────────────────────────────
  +0.000000        -0.000-1.000j
  +0.000+1.000j  +0.000000      

───────────────────────────────────────────────────────
  Gate: Z  (shape (2, 2))
───────────────────────────────────────────────────────
  +1.000000        +0.000000      
  +0.000000        -1.000000      

───────────────────────────────────────────────────────
  Gate: H  (shape (2, 2))
───────────────────────────────────────────────